# MLP

## 感知机

给定输入 $\mathbf{x}$，权重 $\mathbf{w}$，和偏移 $b$，感知机输出：

$$o = \sigma(\langle \mathbf{w}, \mathbf{x} \rangle + b)$$

其中激活函数 $\sigma$ 为：

$$\sigma(x) = \begin{cases} 1 & \text{if } x > 0 \\ -1 & \text{otherwise} \end{cases}$$

### 感知机收敛定理 

假设数据样本在半径 $r$ 的球体内，即对于所有样本 $i$：
$$\|\mathbf{x}_i\|^2 + b_i^2 \le r^2$$

且存在余量 $\rho$ 能将两类数据线性分类，即存在 $\|\mathbf{w}\|^2 + b^2 = 1$ 使得：
$$y_i(\mathbf{x}_i^\top\mathbf{w} + b) \ge \rho$$

**结论：**
感知机保证在以下步数内收敛（即不再发生更新）：
$$\text{更新步数} \le \frac{r^2 + 1}{\rho^2}$$

## 多层感知机

### 多层感知机 (MLP) 结构

- **输入层 (Input Layer)**: $\mathbf{x} \in \mathbb{R}^n$
- **隐藏层 (Hidden Layer)**:
  - 线性变换: $\mathbf{z}_1 = \mathbf{W}_1 \mathbf{x} + \mathbf{b}_1$，其中 $\mathbf{W}_1 \in \mathbb{R}^{m \times n}, \mathbf{b}_1 \in \mathbb{R}^m$
  - 激活输出: $\mathbf{h} = \sigma(\mathbf{z}_1)$
- **输出层 (Output Layer)**:
  - 最终结果: $o = \mathbf{w}_2^T \mathbf{h} + b_2$，其中 $\mathbf{w}_2 \in \mathbb{R}^m, b_2 \in \mathbb{R}$

> **注**：$\sigma$ 通常指 ReLU、Sigmoid 或 Tanh 等非线性激活函数

### 激活函数

Sigmoid: 
$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

Tanh: 
$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

ReLU:
$$\text{ReLU}(x) = \max(0, x)$$

## MLP 从 0 实现

In [1]:
import torch
from torch import nn
from torch.utils import data
import torchvision
from torchvision import transforms

# 准备数据集 
def load_data_fashion_mnist(batch_size, resize=None):
    trans = [transforms.ToTensor()]
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)
    
    mnist_train = torchvision.datasets.FashionMNIST(root="./data", train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(root="./data", train=False, transform=trans, download=True)
    
    return (data.DataLoader(mnist_train, batch_size, shuffle=True, num_workers=4),
            data.DataLoader(mnist_test, batch_size, shuffle=False, num_workers=4))

batch_size = 256
train_iter, test_iter = load_data_fashion_mnist(batch_size)

# 初始化模型参数 
num_inputs, num_outputs, num_hiddens = 784, 10, 256

W1 = nn.Parameter(torch.randn(num_inputs, num_hiddens, requires_grad=True) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens, requires_grad=True))
W2 = nn.Parameter(torch.randn(num_hiddens, num_outputs, requires_grad=True) * 0.01)
b2 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))

params = [W1, b1, W2, b2]

# 定义模型逻辑 
def relu(X):
    """激活函数：ReLU"""
    return torch.max(input=X, other=torch.zeros_like(X))

def net(X):
    """多层感知机网络结构"""
    X = X.reshape((-1, num_inputs)) # 展平图像
    H = relu(X @ W1 + b1)           # 隐藏层运算
    return (H @ W2 + b2)            # 输出层运算

# 使用内置交叉熵损失
loss = nn.CrossEntropyLoss(reduction='none')

# 优化算法与评估
def sgd(params, lr, batch_size):
    """从零实现的随机梯度下降"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

def evaluate_accuracy(net, data_iter):
    """计算模型在指定数据集上的准确率"""
    if isinstance(net, torch.nn.Module):
        net.eval()
    metric = [0.0, 0.0]  # 正确预测数, 样本总数
    with torch.no_grad():
        for X, y in data_iter:
            metric[0] += (net(X).argmax(dim=1) == y).sum().item()
            metric[1] += y.numel()
    return metric[0] / metric[1]

# 训练循环
num_epochs, lr = 10, 0.1

print("开始训练...")
for epoch in range(num_epochs):
    train_loss_sum, train_acc_sum, n = 0.0, 0.0, 0
    
    for X, y in train_iter:
        # 1. 前向传播
        y_hat = net(X)
        l = loss(y_hat, y)
        
        # 2. 反向传播
        l.sum().backward()
        
        # 3. 参数更新
        sgd(params, lr, batch_size)
        
        # 统计数据
        train_loss_sum += l.sum().item()
        train_acc_sum += (y_hat.argmax(dim=1) == y).sum().item()
        n += y.shape[0]
    
    test_acc = evaluate_accuracy(net, test_iter)
    print(f'epoch {epoch + 1}: loss {train_loss_sum/n:.4f}, '
          f'train acc {train_acc_sum/n:.4f}, test acc {test_acc:.4f}')

print("训练完成！")

100.0%
100.0%
100.0%
100.0%

开始训练...


epoch 1: loss 1.0386, train acc 0.6466, test acc 0.7664
epoch 2: loss 0.5995, train acc 0.7903, test acc 0.8097
epoch 3: loss 0.5179, train acc 0.8200, test acc 0.8231
epoch 4: loss 0.4795, train acc 0.8327, test acc 0.8203
epoch 5: loss 0.4544, train acc 0.8401, test acc 0.8344
epoch 6: loss 0.4340, train acc 0.8474, test acc 0.8406
epoch 7: loss 0.4199, train acc 0.8518, test acc 0.8453
epoch 8: loss 0.4058, train acc 0.8570, test acc 0.8486
epoch 9: loss 0.3928, train acc 0.8615, test acc 0.8503
epoch 10: loss 0.3827, train acc 0.8650, test acc 0.8497
训练完成！


## MLP 简洁实现

In [2]:
import torch
from torch import nn
from torch.utils import data
import torchvision
from torchvision import transforms

# 准备数据集
def get_dataloader(batch_size):
    trans = transforms.Compose([transforms.ToTensor()])
    mnist_train = torchvision.datasets.FashionMNIST(root="./data", train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(root="./data", train=False, transform=trans, download=True)
    return (data.DataLoader(mnist_train, batch_size, shuffle=True),
            data.DataLoader(mnist_test, batch_size, shuffle=False))

batch_size = 256
train_iter, test_iter = get_dataloader(batch_size)

# 定义模型
net = nn.Sequential(
    nn.Flatten(),               # 将 (28, 28) 的输入展平为 784 维向量
    nn.Linear(784, 256),        # 隐藏层：784 -> 256
    nn.ReLU(),                  # 激活函数
    nn.Linear(256, 10)          # 输出层：256 -> 10
)

# 初始化权重 
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)

net.apply(init_weights)

# 3. 定义损失函数和优化器
loss = nn.CrossEntropyLoss()    # 内部集成了 Softmax
trainer = torch.optim.SGD(net.parameters(), lr=0.1)

# 4. 训练循环
num_epochs = 10
for epoch in range(num_epochs):
    for X, y in train_iter:
        l = loss(net(X), y)     # 前向传播 + 计算损失
        trainer.zero_grad()     # 清除上一次的梯度
        l.backward()            # 反向传播
        trainer.step()          # 更新模型参数
    
    # 每个 epoch 打印一次进度（这里简化了评估逻辑）
    print(f'epoch {epoch + 1}, loss: {l.item():.4f}')

print("训练完成！")

100.0%
100.0%
100.0%
100.0%


epoch 1, loss: 0.6947
epoch 2, loss: 0.4958
epoch 3, loss: 0.6370
epoch 4, loss: 0.4849
epoch 5, loss: 0.5239
epoch 6, loss: 0.3927
epoch 7, loss: 0.4108
epoch 8, loss: 0.5363
epoch 9, loss: 0.4555
epoch 10, loss: 0.3228
训练完成！
